In [1]:
import pandas as pd

file_path = "VLibrasBD.csv"
df = pd.read_csv(file_path, encoding='utf-8')
df.head(10)

,pt,pgs
0,Orientações específicas para solicitações com ...,ORIENTAÇÃO ESPECÍFICO SOLICITAR OBJETIVO CONTR...
1,Qualquer cidadão interessado em visitar a Bibl...,QUALQUER CIDADÃO INTERESSAR VISITAR BIBLIOTECA...
2,Qualquer cidadão interessado em doar obras par...,QUALQUER CIDADÃO INTERESSAR DOAR&OBJETO OBRA&L...
3,Qualquer cidadão interessado,QUALQUER CIDADÃO INTERESSAR
4,"Qualquer cidadão e/ou empresa brasileira, com ...",QUALQUER CIDADÃO OU EMPRESA BRASIL&PAÍS CPF OU...
5,Qualquer cidadão e cidadã pode participar de c...,QUALQUER CIDADÃO PODER&POSSIBILIDADE PARTICIPA...
6,Qualquer cidadão de qualquer nacionalidade que...,QUALQUER CIDADÃO QUALQUER NACIONALIDADE ACESSA...
7,Qualquer cidadão com uma conta gobr prata ou o...,QUALQUER CIDADÃO CONTA&LOGIN GOBR PRATA OU OUR...
8,Qualquer cidadão com ou sem vínculo com a Univ...,QUALQUER CIDADÃO OU NÃO_TER VÍNCULO UNIVERSIDA...
9,Qualquer cidadão com nível superior completo.,QUALQUER CIDADÃO NÍVEL SUPERIOR COMPLETO [PONTO]


In [2]:
import sys

print("--- STARTING ENVIRONMENT CHECK ---")
print(f"Using Python version: {sys.version}\n")

# --- Checking Main Libraries ---

try:
    import transformers
    print(f"Transformers OK. Installed version: {transformers.__version__}")
except ImportError:
    print("ERROR: 'transformers' library not found.")

try:
    import torch
    print(f"PyTorch (torch) OK. Installed version: {torch.__version__}")
except ImportError:
    print("ERROR: 'torch' library not found.")

try:
    import datasets
    print(f"Datasets OK. Installed version: {datasets.__version__}")
except ImportError:
    print("ERROR: 'datasets' library not found.")

try:
    import accelerate
    print(f"Accelerate OK. Installed version: {accelerate.__version__}")
except ImportError:
    print("ERROR: 'accelerate' library not found.")

try:
    import pandas
    print(f"Pandas OK. Installed version: {pandas.__version__}")
except ImportError:
    print("ERROR: 'pandas' library not found.")

try:
    import sacrebleu
    print(f"Sacrebleu OK. Installed version: {sacrebleu.__version__}")
except ImportError:
    print("ERROR: 'sacrebleu' library not found.")

try:
    import evaluate
    print(f"Evaluate OK. Installed version: {evaluate.__version__}")
except ImportError:
    print("ERROR: 'evaluate' library not found.")


# --- CRITICAL CHECK: GPU Access ---
print("\n--- CHECKING GPU (CUDA) ACCESS ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: PyTorch is installed, but could NOT find your local GPU.")
except Exception as e:
    print(f"ERROR: Failed to check GPU status with PyTorch: {e}")

print("\n--- VERIFICATION COMPLETED ---")

--- STARTING ENVIRONMENT CHECK ---
Using Python version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]

Transformers OK. Installed version: 4.57.1
PyTorch (torch) OK. Installed version: 2.14.1+cu130
Datasets OK. Installed version: 4.4.1
Accelerate OK. Installed version: 1.11.0
Pandas OK. Installed version: 2.3.3
Sacrebleu OK. Installed version: 2.2.1
Evaluate OK. Installed version: 0.4.0

--- CHECKING GPU (CUDA) ACCESS ---
SUCCESS! PyTorch found the GPU: NVIDIA GeForce RTX 5070 Ti

--- VERIFICATION COMPLETED ---


In [3]:
import os

# The script to be executed
generator_script = "gen_variants.py"

# --- Step 1: Check files ---
print("Contents of the working directory BEFORE execution:")
# !ls -l is the Linux 'ls' command (where you are)
!ls -l

# --- Step 2: Execute the script to generate variants ---
print("\nExecuting the gen_variants.py script...")
!python {generator_script}
print("\nScript execution completed!")

# --- Step 3: Verify generated files ---
print("\nContents of the working directory AFTER execution (there should be a 'variants' folder):")
!ls -l


Contents of the working directory BEFORE execution:
total 20464
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br     2948 nov  3 12:11 gen_variants.py
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br 20907773 nov 14  2025 VLibrasBD.csv
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br    37555 nov  3 11:56 vlibrasbd.ipynb

Executing the gen_variants.py script...
--- STARTING DATA PROCESSING ---
Initial total rows: 127349
Rows removed (duplicates): 34
Final total unique rows: 127315
Train Size: 101852
Validation Size: 12731
Test Size: 12732
File generated: pt-pgs.train.csv
File generated: pt-pgs.val.csv
File generated: pt-pgs.test.csv
File generated: pt-pgs.lower.train.csv
File generated: pt-pgs.lower.val.csv
File generated: pt-pgs.lower.test.csv

Process completed successfully!

Script execution completed!

Contents of the working directory AFTER execution (there should be a 'variant

In [4]:
# ==============================================================================
# TRAINING SCRIPT (PHASE 1) - mBART (VLIBRAS PT -> PGS) WITH WORD DROP
# ==============================================================================

import os
import json
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from torch.optim import AdamW
from bitsandbytes.optim import AdamW8bit
from collections import Counter

from transformers import (
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets
import evaluate
from accelerate import Accelerator

# --- 1. Manual Configuration ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "pt"
AUX_LANG = "pgs"
RUN = "Optimized"  

# Hyperparameters
BATCH_SIZE = 2                  
gradient_accumulation_steps = 16
N_EPOCHS = 50                                                   
LR_BODY = 5e-5                                                   
LR_EMBEDDINGS = LR_BODY * 5 # EMBEDDING LEARNING RATE                                            
LR_SCHEDULER = "linear" 
SRC_LANG = 'pt'   
TGT_LANG = 'pgs'
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TRAIN_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.train.csv")
TEST_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.test.csv")
DEV_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.val.csv") 

output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'
os.makedirs(output_dir, exist_ok=True)

# Hardware
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
# ==============================================================================
# 2. VOCABULARY EXTRACTION AND TOKENIZER
# ==============================================================================
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt" 

print(" Extracting VLibras gloss vocabulary...")
df_train_vocab = pd.read_csv(TRAIN_PATH)
df_test_vocab = pd.read_csv(TEST_PATH)
all_glosses_text = pd.concat([df_train_vocab['pgs'], df_test_vocab['pgs']]).str.cat(sep=' ')

gloss_counts = Counter(all_glosses_text.split())
MIN_FREQUENCY = 3 
gloss_vocab = [token for token, count in gloss_counts.items() if count >= MIN_FREQUENCY]
print(f"Total glosses: {len(gloss_counts)}. Useful glosses (>=3): {len(gloss_vocab)}")

print(f" Loading Tokenizer: {MODEL_NAME}")
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(MODEL_NAME)}

# mBART LANGUAGE CONFIGURATION (Universal Portuguese)
tokenizers['pt_X'].src_lang = "pt_XX"
tokenizers['pt_X'].tgt_lang = "pt_XX"
tokenizers['pt_X'].add_tokens(list(gloss_vocab))

# Forced language ID for mBART generation
forced_bos_token_id = tokenizers['pt_X'].lang_code_to_id["pt_XX"]

# ==============================================================================
# 3. DATA PREPARATION (WITH ADAPTED DYNAMIC WORD DROPOUT)
# ==============================================================================
print(" Loading CSVs...")
raw_datasets = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(df_train_vocab),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(df_test_vocab)
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['pt']) and pd.notna(ex['pgs']) and len(str(ex['pt']).strip()) > 0)

def dynamic_word_dropout_and_tokenize(batch):
    p_drop = 0.1                                               # ATTENTION TO DROPOUT
    inputs_dropped = []
    is_training = batch.get('is_train_split', [False])[0] 

    for text in batch["pt"]:
        if is_training and p_drop > 0:
            words = str(text).split()
            kept_words = [w for w in words if random.random() > p_drop]
            if len(kept_words) == 0:
                kept_words = words
            inputs_dropped.append(" ".join(kept_words))
        else:
            inputs_dropped.append(str(text))
            
    targets = [str(ex) for ex in batch["pgs"]]
    
    # mBART Style Tokenization (Using text_target)
    model_inputs = tokenizers['pt_X'](inputs_dropped, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizers['pt_X'](text_target=targets, max_length=MAX_TGT_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

raw_datasets['train'] = raw_datasets['train'].add_column("is_train_split", [True] * len(raw_datasets['train']))
raw_datasets['validation'] = raw_datasets['validation'].add_column("is_train_split", [False] * len(raw_datasets['validation']))
raw_datasets['test'] = raw_datasets['test'].add_column("is_train_split", [False] * len(raw_datasets['test']))

print(" Configuring Dynamic Tokenization...")
raw_datasets.set_transform(dynamic_word_dropout_and_tokenize)

# ==============================================================================
# 4. MODEL & DIFFERENTIAL OPTIMIZER
# ==============================================================================
best_bleu = 0.0
checkpoint_config = os.path.join(output_dir, "config.json")

if os.path.exists(checkpoint_config):
    print(f"\n CHECKPOINT FOUND. Resuming from: {output_dir}")
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    try:
        with open(os.path.join(output_dir, 'stats.json'), 'r') as f:
            best_bleu = max(json.load(f).get('bleu', [0]))
    except: pass
else:
    print(f"\n STARTING FROM SCRATCH with '{MODEL_NAME}'...")
    config = AutoConfig.from_pretrained(MODEL_NAME)
    config.dropout = 0.3                #   ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!                                         
    config.attention_dropout = 0.1      #   ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, config=config)
    model.resize_token_embeddings(len(tokenizers['pt_X']))

# Differential LR 
embedding_params = model.get_input_embeddings().parameters()
embedding_param_ids = {id(p) for p in embedding_params}
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]

optimizer_grouped_parameters = [
    {"params": body_params, "lr": LR_BODY},
    {"params": model.get_input_embeddings().parameters(), "lr": LR_EMBEDDINGS},
]
optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=0.05)

data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model)
train_dataloader = DataLoader(raw_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(raw_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

num_training_steps = N_EPOCHS * len(train_dataloader) #ATTENTION TO WARM-UP!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
lr_scheduler = get_scheduler(LR_SCHEDULER, optimizer=optimizer, num_warmup_steps=int(num_training_steps*0.05), num_training_steps=num_training_steps)

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id)
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True)
    return [p.strip() for p in decoded_preds], [[l.strip()] for l in decoded_labels]

# ==============================================================================
# 5. TRAINING LOOP (WITH CORRECTED LABEL SMOOTHING)
# ==============================================================================
patience = 10 # CHANGE LATER                                                                                        
patience_counter = 0
progress_bar = tqdm(range(num_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')
stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100) #ATTENTION TO LABEL SMOOTHING!!!!!!!!!!!!!!!!!!!!
    
    for batch in train_dataloader:
        outputs = model(**batch)
        
        logits = outputs.logits
        labels = batch["labels"]
        
        # OPTIMIZED MANUAL LOSS CALCULATION TO APPLY SMOOTHING WITHOUT OOM
        loss = loss_fct(logits.transpose(1, 2), labels)
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    # --- VALIDATION ---
    model.eval()
    metric = evaluate.load("sacrebleu") 
    
    for batch in tqdm(eval_dataloader, desc=f"Validation Epoch {epoch+1}", leave=False):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=5 ######################### ATTENTION TO NUM_BEAMS
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        preds_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(preds_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    # MAINTAINED PURIST EVALUATION
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    b1, b2, b3, b4 = results['precisions']
    print(f"\n--- Epoch {epoch+1} ---")
    print(f" Loss: {avg_loss:.4f} | SacreBLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    print(f"   Isolated BLEUs -> BLEU-1: {b1:.2f} | BLEU-2: {b2:.2f} | BLEU-3: {b3:.2f} | BLEU-4: {b4:.2f}")
    
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f" New Record! Saving model...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f: json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(" Early stopping.")
        break

writer.close()

 Extracting VLibras gloss vocabulary...
Total glosses: 52625. Useful glosses (>=3): 19049
 Loading Tokenizer: facebook/mbart-large-50-many-to-many-mmt
 Loading CSVs...


Filter:   0%|          | 0/101852 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12731 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12732 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/101852 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/12731 [00:00<?, ? examples/s]

... (4 outputs left out)

Validation Epoch 1:   0%|          | 0/6366 [00:00<?, ?it/s]


--- Epoch 1 ---
 Loss: 5.9562 | SacreBLEU: 2.15 (Best: 0.00)
   Isolated BLEUs -> BLEU-1: 29.51 | BLEU-2: 4.93 | BLEU-3: 1.35 | BLEU-4: 0.50
 New Record! Saving model...


/home/j170860@rsrch.cepetro.unicamp.br/miniconda3/envs/nllb_env/lib/python3.10/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


KeyboardInterrupt: 

In [6]:
# ==============================================================================
# TRAINING SCRIPT - PHASE 2 (VLIBRAS mBART EXPERT FINE-TUNING)
# ==============================================================================

import os
import json
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from bitsandbytes.optim import AdamW8bit

from transformers import (
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets                             # Provides dataset handling structures
datasets.config.TORCHVISION_AVAILABLE = False
import sys
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]
from accelerate import Accelerator
import evaluate

# --- 1. PATH CONFIGURATION ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "pt"
AUX_LANG = "pgs"

# POINTS TO THE FOLDER GENERATED IN PHASE 1
PREV_RUN_DIR = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-runOptimized' # ATTENTION HERE
RUN = "Optimized_Run" #ATTENTION HERE 
 
BATCH_SIZE = 2                  
gradient_accumulation_steps = 4 
N_EPOCHS = 40                     
LR_BODY = 5e-6                    # Drastic cooling in the model body
LR_EMBEDDINGS = 2e-5              # Moderate cooling in the glosses
LR_SCHEDULER = "linear" 
SRC_LANG = 'pt'   
TGT_LANG = 'pgs'
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TRAIN_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.train.csv") 
TEST_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.test.csv")
DEV_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.val.csv") 

output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'
os.makedirs(output_dir, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==============================================================================
# 2. DATA PREPARATION (STATIC FUNCTION - NO DROPOUT)
# ==============================================================================
print(f" Loading Tokenizer from Phase 1: {PREV_RUN_DIR}")
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(PREV_RUN_DIR)}

# FORCING MANDATORY mBART LANGUAGES
tokenizers['pt_X'].src_lang = "pt_XX"
tokenizers['pt_X'].tgt_lang = "pt_XX"

print(" Loading CSVs...")
raw_datasets = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(pd.read_csv(TRAIN_PATH)),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(pd.read_csv(TEST_PATH))
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['pt']) and pd.notna(ex['pgs']) and len(str(ex['pt']).strip()) > 0)
raw_datasets = raw_datasets.map(
    lambda ex: {"translation": {SRC_LANG: str(ex[SRC_LANG]), TGT_LANG: str(ex[TGT_LANG])}},
    remove_columns=[SRC_LANG, TGT_LANG]
)

def preprocess_function(examples):
    inputs = [str(ex[SRC_LANG]) for ex in examples["translation"]]
    targets = [str(ex[TGT_LANG]) for ex in examples["translation"]]
    
    # Clean mBART logic
    model_inputs = tokenizers['pt_X'](inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizers['pt_X'](text_target=targets, max_length=MAX_TGT_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(" Tokenizing...")
tokenized_datasets = raw_datasets.map(
    preprocess_function, 
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
tokenized_datasets.set_format("torch")

# ==============================================================================
# 3. MODEL & DIFFERENTIAL OPTIMIZER
# ==============================================================================
print(f"\n Loading Model from Phase 1: {PREV_RUN_DIR}")

config = AutoConfig.from_pretrained(PREV_RUN_DIR)
# Reduced dropout, since data is clean and the model should be confident
config.dropout = 0.1                                                                    
model = AutoModelForSeq2SeqLM.from_pretrained(PREV_RUN_DIR, config=config)

# Separation for Phase 2 Differential LR
embedding_params = model.get_input_embeddings().parameters()
embedding_param_ids = {id(p) for p in embedding_params}
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]

optimizer_grouped_parameters = [
    {"params": body_params, "lr": LR_BODY},
    {"params": model.get_input_embeddings().parameters(), "lr": LR_EMBEDDINGS},
]

optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=0.01)                                
data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model)

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(tokenized_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

num_training_steps = N_EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(LR_SCHEDULER, optimizer=optimizer, num_warmup_steps=int(num_training_steps*0.1), num_training_steps=num_training_steps)
metric = evaluate.load("sacrebleu")

forced_bos_token_id = tokenizers['pt_X'].lang_code_to_id["pt_XX"]

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id)
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True)
    return [p.strip() for p in decoded_preds], [[l.strip()] for l in decoded_labels]

# ==============================================================================
# 4. TRAINING LOOP (PHASE 2 WITH SMOOTHING)
# ==============================================================================
best_bleu = 0.0
patience = 10                                                                                                                                
patience_counter = 0

print(f"\n START PHASE 2 | Epochs: {N_EPOCHS}")
progress_bar = tqdm(range(num_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')
stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    
    # REINTRODUCTION OF LABEL SMOOTHING TO PREVENT OVERFITTING
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100) 
    
    for batch in train_dataloader:
        outputs = model(**batch)
        
        logits = outputs.logits
        labels = batch["labels"]
        
        # SAFE MANUAL CALCULATION FOR RTX MEMORY
        loss = loss_fct(logits.transpose(1, 2), labels) 
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    # --- PURIST VALIDATION ---
    model.eval()
    print(f"\n--- Epoch {epoch+1} ---")
    
    for batch in tqdm(eval_dataloader, desc="Validation", leave=False):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id # MANDATORY FOR mBART
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    # STRICT EVALUATION: Exactly as in the final test
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    b1, b2, b3, b4 = results['precisions']
    
    print(f" Loss: {avg_loss:.4f} |  Overall BLEU-4: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    print(f"   B1: {b1:.1f} | B2: {b2:.1f} | B3: {b3:.1f} | B4: {b4:.1f}")
    
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f" Record! Saving Expert Phase...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f: json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(" Early stopping of the Expert Phase.")
        break

writer.close()
print(" Phase 2 successfully completed! You can run the inference script to measure the leap.")

 Loading Tokenizer from Phase 1: runs/pt-pgs-lower-runOptimized
 Loading CSVs...


Filter:   0%|          | 0/101852 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12731 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12732 [00:00<?, ? examples/s]

Map:   0%|          | 0/101852 [00:00<?, ? examples/s]

Map:   0%|          | 0/12731 [00:00<?, ? examples/s]

... (32 outputs left out)

 Loss: 1.8388 |  Overall BLEU-4: 58.00 (Best: 57.71)
   B1: 82.0 | B2: 64.9 | B3: 51.7 | B4: 41.1
 Record! Saving Expert Phase...

--- Epoch 13 ---


Validation:   0%|          | 0/6366 [00:00<?, ?it/s]

 Loss: 1.8130 |  Overall BLEU-4: 58.46 (Best: 58.00)
   B1: 82.3 | B2: 65.4 | B3: 52.3 | B4: 41.6
 Record! Saving Expert Phase...


KeyboardInterrupt: 

In [7]:
# ==============================================================================
# INFERENCE AND EVALUATION SCRIPT - mBART (VLIBRASBD)
# BLEU (1 to 4), SACREBLEU AND ROUGE-L CALCULATION
# ==============================================================================
import os
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Mandatory imports for standardized metrics
import sacrebleu
from sacrebleu.metrics import BLEU
import evaluate

# Clears VRAM residue from previous trainings
torch.cuda.empty_cache()

# --- 1. PATH AND VARIABLE CONFIGURATION ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "pt"
AUX_LANG = "pgs"

# POINTS TO THE CORRECT VLIBRAS PHASE FOLDER
RUN = "Optimized_Run" 

SRC_LANG = 'pt'
TGT_LANG = 'pgs'
VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TEST_PATH = os.path.join(DATASET_PATH, f"pt-pgs{VARIANT}.test.csv")
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'

# --- 2. INFERENCE PARAMETERS ---
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"\n STARTING INFERENCE (VLIBRASBD)")
print(f" Looking for model in: {output_dir}")
print(f" Reading test data from: {TEST_PATH}")

# --- 3. MODEL AND TOKENIZER LOADING ---
if not os.path.exists(output_dir):
    raise FileNotFoundError(f" Model folder not found: {output_dir}. Check the RUN name.")

# Loads the model in evaluation mode (deterministic)
best_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(output_dir)

#  MANDATORY mBART CONFIGURATIONS FOR VLIBRAS 
best_tokenizer.src_lang = "pt_XX"
forced_bos_token_id = best_tokenizer.lang_code_to_id["pt_XX"]

best_model.eval()

# --- 4. DATA READING AND PREPARATION ---
df_test_final = pd.read_csv(TEST_PATH)
df_test_final = df_test_final.dropna(subset=[SRC_LANG, TGT_LANG])

preds_final = []
refs_final = []

# --- 5. GENERATION LOOP ---
for index, row in tqdm(df_test_final.iterrows(), total=len(df_test_final), desc="Generating Translations"):
    src_text = str(row[SRC_LANG])
    tgt_text = str(row[TGT_LANG])
    
    inputs = best_tokenizer(src_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(DEVICE)
    
    with torch.no_grad():
        outputs = best_model.generate(
            **inputs, 
            max_length=MAX_TGT_LENGTH,
            forced_bos_token_id=forced_bos_token_id, 
            num_beams=5,                  # Aligned with testing heuristics. CHECK HERE!!!!!!!!!!!!!!
            early_stopping=True           # Stops generation if the model confirms the end of the sentence
        )
        
    pred_str = best_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    preds_final.append(pred_str)
    # ATTENTION: We save the pure string, not as a list, for later formatting
    refs_final.append(tgt_text) 

# --- 6. METRICS CALCULATION ---
print("\n Calculating purist metrics...")

# Formats required by each library
refs_transposed = [refs_final]  # Format required by SacreBLEU
refs_flat = refs_final          # Format required by ROUGE

results = {}

# A) SacreBLEU (Canonical) - TOKENIZE NONE MANDATORY FOR GLOSSES
sb = sacrebleu.corpus_bleu(preds_final, refs_transposed, lowercase=True, tokenize='none')
results['SacreBLEU'] = sb.score

# B) Granular BLEU 1, 2, 3, 4
results['BLEU-1'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=1).corpus_score(preds_final, refs_transposed).score
results['BLEU-2'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=2).corpus_score(preds_final, refs_transposed).score
results['BLEU-3'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=3).corpus_score(preds_final, refs_transposed).score
results['BLEU-4'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=4).corpus_score(preds_final, refs_transposed).score

# C) ROUGE-L (Recall-Oriented)
try:
    rouge = evaluate.load("rouge")
    rouge_res = rouge.compute(predictions=preds_final, references=refs_flat)
    results['ROUGE-L'] = rouge_res['rougeL'] * 100 
except Exception as e:
    print(f" Warning: Could not calculate ROUGE ({e})")
    results['ROUGE-L'] = 0.0

# --- 7. RESULTS DISPLAY ---
print("\n" + "="*40)
print(f" FINAL PERFORMANCE REPORT (VLIBRASBD)")
print("="*40)
print(f"{'Metric':<15} | {'Score':<10}")
print("-" * 28)
print(f"{'BLEU-1 ':<15} | {results['BLEU-1']:.2f}")
print(f"{'BLEU-2 ':<15} | {results['BLEU-2']:.2f}")
print(f"{'BLEU-3 ':<15} | {results['BLEU-3']:.2f}")
print(f"{'BLEU-4 ':<15} | {results['BLEU-4']:.2f}")
print("-" * 28)
print(f"{'SacreBLEU':<15} | {results['SacreBLEU']:.2f}")
print(f"{'ROUGE-L':<15} | {results['ROUGE-L']:.2f}")
print("="*40)

# --- 8. DETAILED SAVING ---
df_result_file = pd.DataFrame({
    'pt': df_test_final['pt'], 
    'pgs_ref': refs_final, 
    'pgs_pred': preds_final
})

csv_out = os.path.join(output_dir, 'final_metrics_evaluation.csv')
df_result_file.to_csv(csv_out, index=False)
print(f"\nDetailed file with translations saved at: {csv_out}")


 STARTING INFERENCE (VLIBRASBD)
 Looking for model in: runs/pt-pgs-lower-runOptimized_Run
 Reading test data from: ./variants/pt-pgs.lower.test.csv


Generating Translations:   0%|          | 0/12732 [00:00<?, ?it/s]


 Calculating purist metrics...

 FINAL PERFORMANCE REPORT (VLIBRASBD)
Metric          | Score     
----------------------------
BLEU-1          | 81.41
BLEU-2          | 72.28
BLEU-3          | 64.25
BLEU-4          | 57.03
----------------------------
SacreBLEU       | 57.03
ROUGE-L         | 87.60

Detailed file with translations saved at: runs/pt-pgs-lower-runOptimized_Run/final_metrics_evaluation.csv


In [ ]:
##########################################################################################################################################################